In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df=pd.read_csv("loan_approval_data.csv")

In [ ]:
df.head()
df.info()
df.isnull().sum()
# df.describe()

## Handle missing values

In [ ]:
categorical_cols=df.select_dtypes(include=["str"]).columns #select_dtype:used to select coulumn depending upon datatype
numerical_cols=df.select_dtypes(include=["number"]).columns

In [ ]:
from sklearn.impute import SimpleImputer

num_imp=SimpleImputer(strategy="mean")
df[numerical_cols]=num_imp.fit_transform(df[numerical_cols])

cat_imp=SimpleImputer(strategy="most_frequent")
df[categorical_cols]=cat_imp.fit_transform(df[categorical_cols])

In [ ]:
df.head()
df.isnull().sum()

# EDA- exploratory data analysis

In [ ]:
#how balanced our classes are?

classes_count=df["Loan_Approved"].value_counts()

plt.pie(classes_count,labels=["no","yes"],autopct="%1.1f%%")
plt.title("is loan approved or not?")

In [ ]:
classes_count

In [ ]:
#analize categorical data
# gender_count=df["Gender"].value_counts()
# sns.barplot(gender_count)

# education=df["Education_Level"].value_counts()
# sns.barplot(education)

# employer_cat=df["Employer_Category"].value_counts()
# sns.barplot(employer_cat)

# Property=df["Property_Area"].value_counts()
# sns.barplot(Property)

# Loan_Purpose=df["Loan_Purpose"].value_counts()
# sns.barplot(Loan_Purpose)

# status=df["Employment_Status"].value_counts()
# sns.barplot(status)

life_status=df["Marital_Status"].value_counts()
sns.barplot(life_status)

In [ ]:
#analyze income:

sns.histplot(
    data=df,
    x="Applicant_Income",
    bins=20
    
)

In [ ]:
sns.histplot(
    data=df,
    x="Coapplicant_Income",
    bins=20
    
)

In [ ]:
#to detect the outliers use -box plots

sns.boxplot(
    data=df,
    x="Loan_Approved",
    y="Applicant_Income"
)

In [ ]:
fig,axes=plt.subplots(2,3)

sns.boxplot(ax=axes[0,0],data=df,x="Loan_Approved",y="Applicant_Income")
sns.boxplot(ax=axes[0,1],data=df,x="Loan_Approved",y="Credit_Score")
sns.boxplot(ax=axes[0,2],data=df,x="Loan_Approved",y="DTI_Ratio")
sns.boxplot(ax=axes[1,0],data=df,x="Loan_Approved",y="Savings")
sns.boxplot(ax=axes[1,1],data=df,x="Loan_Approved",y="Age")
sns.boxplot(ax=axes[1,2],data=df,x="Loan_Approved",y="Loan_Amount")

plt.tight_layout()

In [ ]:
#credit scpre with loan approved

sns.histplot(
    data=df,
    x="Credit_Score",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
    
)

In [ ]:
sns.histplot(
    data=df,
    x="Applicant_Income",
    hue="Loan_Approved",
    bins=20,
    multiple="dodge"
    
)

In [ ]:
#remove applicant id

df=df.drop("Applicant_ID",axis=1)

In [ ]:
df

# Encoding

In [ ]:
df.head()
df.columns
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

le=LabelEncoder()

df["Education_Level"]=le.fit_transform(df["Education_Level"])
df["Loan_Approved"]=le.fit_transform(df["Loan_Approved"])
       

In [ ]:
df.head()

In [ ]:
cols=["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]


ohe=OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")

#gives us a 2d array:
encoded=ohe.fit_transform(df[cols])

#convert 2d frame into dataframe:
encoded_df=pd.DataFrame(encoded,columns=ohe.get_feature_names_out(cols),index=df.index)

df=pd.concat([df.drop(columns=cols),encoded_df],axis=1)

In [ ]:
df.head()

In [ ]:
df.info()

# correlation heatmap

In [ ]:
num_cols=df.select_dtypes(include="number")
corr_matrix=num_cols.corr()



In [ ]:
num_cols.corr()["Loan_Approved"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(15,8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

# train test split + feature scaling

In [ ]:
X=df.drop("Loan_Approved",axis=1)
y=df["Loan_Approved"]

In [ ]:
X.head()

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_test.head()

In [ ]:
#feature scaling
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train_scaled

# Train and evaluate models

In [ ]:
#logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

log_model=LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred=log_model.predict(X_test_scaled)

#evaluation
print("logistic regression model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))


In [ ]:
#KNN classifier:
from sklearn.neighbors import KNeighborsClassifier

knn_model=KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled,y_train)

y_pred=knn_model.predict(X_test_scaled)
 
#evaluation
print("knn model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))


In [ ]:
#naive byes:
from sklearn.naive_bayes import GaussianNB

nb_model=GaussianNB()
nb_model.fit(X_train_scaled,y_train)

y_pred=nb_model.predict(X_test_scaled)
 
#evaluation
print("naive_byes model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))


# best model on the basis of precision--> NAIVE BYES

# feature engineering

In [ ]:
#add or transform features
df["DTI_ratio_sq"]=df["DTI_Ratio"]**2
df["credit_score_sq"]=df["Credit_Score"]**2

# df["Applicanta_income_log"]=np.log1p(df["Applicant_Income"])

X=df.drop(columns=["Loan_Approved","Credit_Score","DTI_Ratio"])
y=df["Loan_Approved"]

#train test split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#snaling
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
#logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

log_model=LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred=log_model.predict(X_test_scaled)

#evaluation
print("logistic regression model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))


In [ ]:
#KNN classifier:
from sklearn.neighbors import KNeighborsClassifier

knn_model=KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled,y_train)

y_pred=knn_model.predict(X_test_scaled)
 
#evaluation
print("knn model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))


In [ ]:
#naive byes:
from sklearn.naive_bayes import GaussianNB

nb_model=GaussianNB()
nb_model.fit(X_train_scaled,y_train)

y_pred=nb_model.predict(X_test_scaled)
 
#evaluation
print("naive_byes model")
print("precision",precision_score(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))
print("accuracy",accuracy_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))
